In [ ]:
from IPython.display import HTML, display

display(HTML("""
<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 60%, #0f3460 100%);
            border-radius: 12px; padding: 40px 40px 30px 40px; margin: 10px 0 30px 0;
            font-family: 'Helvetica Neue', Arial, sans-serif;">
  <div style="color:#a0aec0; font-size:0.8em; letter-spacing:2px; text-transform:uppercase; margin-bottom:12px;">
    LLM Lab — Section 02
  </div>
  <h1 style="color:white; font-size:2.2em; margin:0 0 10px 0; font-weight:700; line-height:1.2; border:none;">
    Tokenization and Embeddings<br>
    <span style="color:#63b3ed;">BPE, Subwords, and Vector Spaces</span>
  </h1>
  <p style="color:#cbd5e0; font-size:1em; margin:16px 0 24px 0; max-width:600px; line-height:1.6;">
    How text becomes numbers — the tokenizer splits text into tokens,
    then embeddings map those tokens to vectors where meaning is geometry.
  </p>
  <div style="display:flex; gap:16px; flex-wrap:wrap;">
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">Tokenization</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">BPE</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">Embeddings</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">Vector Spaces</span>
    <span style="background:rgba(255,255,255,0.1); color:#e2e8f0; padding:6px 14px; border-radius:20px; font-size:0.85em;">⏱ ~25 min</span>
  </div>
</div>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">💻 Step 1: Environment Check</h2>
</div>

First — verify the environment is ready.

In [ ]:
!python3 ../../scripts/setup_check.py


<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 Step 2: Server Discovery &amp; Warmup</h2>
</div>

Discover running MLX servers and warm them up in parallel.

In [ ]:
# Discover running MLX servers and warm up in parallel
from openai import OpenAI
import time
import concurrent.futures
from IPython.display import HTML, display

# Import shared helpers (includes discover_servers)
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../scripts").resolve()))
import notebook_helpers

# Discover servers (reads model ID from process command line, not /v1/models)
print("Discovering MLX servers...", flush=True)
MODELS, clients = notebook_helpers.discover_servers()

if not MODELS:
    raise RuntimeError("No MLX servers found! Start at least one on ports 8800-8802.")

for m in MODELS:
    print(f"  Port {m['port']}: {m['label']} ({m['model']})", flush=True)

# Warm up all discovered models in parallel
def warmup(m):
    t0 = time.time()
    try:
        clients[m["label"]].chat.completions.create(
            model=m["model"],
            messages=[{"role": "user", "content": "hi"}],
            max_tokens=1,
        )
        return m["label"], time.time() - t0, True
    except Exception as e:
        return m["label"], time.time() - t0, False

print(f"\nWarming up {len(MODELS)} models in parallel...", flush=True)
with concurrent.futures.ThreadPoolExecutor(max_workers=len(MODELS)) as ex:
    warmup_results = list(ex.map(warmup, MODELS))

# Display status table
rows = ""
for label, elapsed, ok in warmup_results:
    status = "\u2713" if ok else "\u2717"
    color = "#16a34a" if ok else "#dc2626"
    m = next(m for m in MODELS if m["label"] == label)
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:{m['color']}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{label}</td>"
        f"<td style='padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;'>{m['port']}</td>"
        f"<td style='padding:6px 12px; color:#374151; font-size:0.85em; border-bottom:1px solid #e5e7eb;'>{m['model']}</td>"
        f"<td style='padding:6px 12px; color:{color}; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{status}</td>"
        f"<td style='padding:6px 12px; color:#111827; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{elapsed:.1f}s</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:400px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Port</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Model ID</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Status</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Warmup</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
"""))
print(f"All {len(MODELS)} models ready!")

notebook_helpers.init(MODELS, clients)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧰 Step 3: Load Helpers</h2>
</div>

Import shared utility functions.

In [ ]:
from notebook_helpers import strip_think, compare_models, show_metrics_table
print("Helpers loaded.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🔤 Step 4: What Is Tokenization?</h2>
</div>

Every LLM has a front door: the **tokenizer**. It breaks raw text into discrete integer IDs the model can process. Think of it like an ADC — sampling continuous text into discrete values.

```
Raw Text           Tokenizer              Embedding Layer         Transformer
"Hello world" ──→ [15496, 1917] ──→ [[0.12, -0.03, ...],  ──→  Attention,
                                     [0.08,  0.41, ...]]       FFN, etc.
```

Let's ask the models to explain it.

In [ ]:
# Ask models to explain tokenization
results = compare_models(
    "Explain BPE (Byte Pair Encoding) tokenization in 3 sentences. "
    "Include a concrete example showing how a word gets split into subword tokens.",
)
show_metrics_table(results)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🔧 Step 5: BPE Merge Walkthrough</h2>
</div>

BPE starts with individual characters, counts adjacent pairs, and merges the most frequent. Let's watch it happen step by step.

In [ ]:
# BPE merge walkthrough — pure Python, no dependencies
from collections import Counter
from IPython.display import HTML, display

corpus = "low lower newest widest"
# Start with characters (space-separated words with end-of-word marker)
words = corpus.split()
tokenized = [list(w) + ["</w>"] for w in words]

rows = ""
rows += (
    f"<tr><td style='padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;'>Initial</td>"
    f"<td style='padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;'>—</td>"
    f"<td style='padding:6px 12px; color:#111827; font-family:monospace; border-bottom:1px solid #e5e7eb;'>{' | '.join([' '.join(w) for w in tokenized])}</td></tr>"
)

for step in range(6):
    # Count all adjacent pairs
    pairs = Counter()
    for w in tokenized:
        for i in range(len(w) - 1):
            pairs[(w[i], w[i+1])] += 1
    if not pairs:
        break
    # Merge most frequent
    best = pairs.most_common(1)[0]
    (a, b), count = best
    merged = a + b
    # Apply merge to all words
    new_tokenized = []
    for w in tokenized:
        new_w = []
        i = 0
        while i < len(w):
            if i < len(w) - 1 and w[i] == a and w[i+1] == b:
                new_w.append(merged)
                i += 2
            else:
                new_w.append(w[i])
                i += 1
        new_tokenized.append(new_w)
    tokenized = new_tokenized
    rows += (
        f"<tr><td style='padding:6px 12px; color:#374151; border-bottom:1px solid #e5e7eb;'>Merge {step+1}</td>"
        f"<td style='padding:6px 12px; color:#2563eb; font-weight:bold; font-family:monospace; border-bottom:1px solid #e5e7eb;'>({a}, {b}) → {merged} (count={count})</td>"
        f"<td style='padding:6px 12px; color:#111827; font-family:monospace; border-bottom:1px solid #e5e7eb;'>{' | '.join([' '.join(w) for w in tokenized])}</td></tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:sans-serif; width:100%;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Step</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Merge Rule</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Result</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">Corpus: \"{corpus}\" — BPE compresses frequent patterns first, just like Huffman coding.</div>
"""))
print(f"Final vocabulary pieces: {sorted(set(t for w in tokenized for t in w))}")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📊 Step 6: Vocabulary Size Tradeoffs</h2>
</div>

Larger vocabulary = fewer tokens per text = faster inference, but more memory for the embedding table. It's like choosing ADC resolution.

In [ ]:
# Vocabulary size tradeoff visualization
from IPython.display import HTML, display

# Embedding table size: vocab_size × d_model × bytes_per_param
d_model = 4096  # typical for modern LLMs
bytes_per = 2   # fp16

configs = [
    ("GPT-2", 50257, "English-centric"),
    ("GPT-4 (cl100k)", 100256, "Better multilingual + code"),
    ("GPT-4o (o200k)", 200019, "Doubled vocab, best multilingual"),
    ("Qwen3", 151936, "Strong multilingual + code"),
]

rows = ""
for name, vocab, note in configs:
    size_mb = (vocab * d_model * bytes_per) / 1e6
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:#374151; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{name}</td>"
        f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{vocab:,}</td>"
        f"<td style='padding:6px 12px; color:#2563eb; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{size_mb:.0f} MB</td>"
        f"<td style='padding:6px 12px; color:#6b7280; border-bottom:1px solid #e5e7eb;'>{note}</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; min-width:500px;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Tokenizer</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Vocab Size</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Embedding Table (fp16)</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Notes</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">Embedding table = vocab_size × {d_model} (d_model) × 2 bytes (fp16). Larger vocab = more memory but fewer tokens per text.</div>
"""))

print("\nThe tradeoff: bigger vocab means shorter sequences (faster) but larger embedding tables (more RAM).")
print(f"Doubling vocab from 100K to 200K adds {(100000 * d_model * bytes_per) / 1e6:.0f} MB but can cut token counts 10-20% for non-English text.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🏷️ Step 7: Special Tokens &amp; Chat Templates</h2>
</div>

Special tokens control model behavior — they're added explicitly, never generated by BPE merges.

In [ ]:
# Visualize chat template structure
from IPython.display import HTML, display

display(HTML("""
<div style="background:#1a1a2e; border-radius:8px; padding:20px; margin:10px 0; font-family:monospace; color:#e2e8f0; font-size:0.9em; line-height:1.8;">
  <div style="color:#a0aec0; margin-bottom:12px;">Chat Template Structure (Qwen3.5 format):</div>
  <span style="background:#2563eb; color:white; padding:2px 8px; border-radius:4px;">&lt;|im_start|&gt;</span>
  <span style="color:#f59e0b;">system</span><br>
  <span style="color:#cbd5e0; margin-left:20px;">You are a helpful assistant.</span><br>
  <span style="background:#dc2626; color:white; padding:2px 8px; border-radius:4px;">&lt;|im_end|&gt;</span><br>
  <br>
  <span style="background:#2563eb; color:white; padding:2px 8px; border-radius:4px;">&lt;|im_start|&gt;</span>
  <span style="color:#f59e0b;">user</span><br>
  <span style="color:#cbd5e0; margin-left:20px;">What is UART?</span><br>
  <span style="background:#dc2626; color:white; padding:2px 8px; border-radius:4px;">&lt;|im_end|&gt;</span><br>
  <br>
  <span style="background:#2563eb; color:white; padding:2px 8px; border-radius:4px;">&lt;|im_start|&gt;</span>
  <span style="color:#f59e0b;">assistant</span><br>
  <span style="color:#16a34a; margin-left:20px;">← model generates here</span>
</div>

<div style="background:#f9fafb; border:1px solid #d1d5db; border-radius:8px; padding:16px; margin:10px 0;">
  <div style="font-weight:bold; color:#1e3a5f; margin-bottom:8px;">Common Special Tokens:</div>
  <table style="font-family:monospace; font-size:0.85em;">
    <tr><td style="padding:4px 12px; color:#2563eb;">&lt;|im_start|&gt;</td><td style="color:#374151;">Start of message block</td></tr>
    <tr><td style="padding:4px 12px; color:#dc2626;">&lt;|im_end|&gt;</td><td style="color:#374151;">End of message block</td></tr>
    <tr><td style="padding:4px 12px; color:#16a34a;">&lt;|endoftext|&gt;</td><td style="color:#374151;">End of sequence (EOS)</td></tr>
    <tr><td style="padding:4px 12px; color:#f59e0b;">&lt;think&gt;...&lt;/think&gt;</td><td style="color:#374151;">Reasoning block (Qwen3.5)</td></tr>
  </table>
</div>
"""))

print("Special tokens are never generated by BPE merges — they're added explicitly by the tokenizer.")
print("If you accidentally include <|endoftext|> in training data, the model learns to stop mid-sentence.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🌍 Step 8: Multilingual Fertility</h2>
</div>

**Fertility** = tokens per word. English gets ~1.3 tokens/word. Other languages get far more — meaning higher costs, shorter effective context windows, and slower inference.

In [ ]:
# Multilingual fertility comparison — ask models to count their own tokens
# Since we can't access the tokenizer directly, we'll use the models' usage stats
from IPython.display import HTML, display

texts = {
    "English":  "The weather is nice today and I feel happy.",
    "Spanish":  "El clima est\u00e1 agradable hoy y me siento feliz.",
    "Japanese": "\u4eca\u65e5\u306f\u3044\u3044\u5929\u6c17\u3067\u3059\u3002\u5e78\u305b\u306a\u6c17\u5206\u3067\u3059\u3002",
    "Arabic":   "\u0627\u0644\u0637\u0642\u0633 \u0644\u0637\u064a\u0641 \u0627\u0644\u064a\u0648\u0645 \u0648\u0623\u0646\u0627 \u0633\u0639\u064a\u062f.",
    "Code":     "def init_gpio(port, pin, mode):\n    GPIO[port].MODER &= ~(0x3 << (pin*2))",
}

# Use the first available model to measure token counts via the API
target = MODELS[0]
print(f"Measuring token counts with {target['label']} on port {target['port']}...", flush=True)

rows = ""
for lang, text in texts.items():
    response = clients[target["label"]].chat.completions.create(
        model=target["model"],
        messages=[{"role": "user", "content": text}],
        max_tokens=1,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    prompt_tokens = response.usage.prompt_tokens
    # Subtract ~5 tokens for chat template overhead
    content_tokens = max(prompt_tokens - 5, 1)
    words = len(text.split())
    fertility = content_tokens / words if words > 0 else 0
    bar_width = min(int(fertility * 40), 200)
    bar_color = "#16a34a" if fertility < 2 else "#f59e0b" if fertility < 3 else "#dc2626"
    rows += (
        f"<tr>"
        f"<td style='padding:6px 12px; color:#374151; font-weight:bold; border-bottom:1px solid #e5e7eb;'>{lang}</td>"
        f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{text[:40]}{'...' if len(text) > 40 else ''}</td>"
        f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>{words}</td>"
        f"<td style='padding:6px 12px; color:#111827; border-bottom:1px solid #e5e7eb;'>~{content_tokens}</td>"
        f"<td style='padding:6px 12px; border-bottom:1px solid #e5e7eb;'>"
        f"<div style='background:{bar_color}; width:{bar_width}px; height:14px; border-radius:3px; display:inline-block;'></div>"
        f" <span style='color:#111827; font-weight:bold;'>{fertility:.1f}</span></td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:monospace; width:100%;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Language</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Text</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Words</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Tokens</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Fertility (tok/word)</th>
  </tr></thead>
  <tbody>{rows}</tbody>
</table>
<div style="color:#6b7280; font-size:0.8em; margin-top:4px;">Higher fertility = more tokens per word = higher API costs, shorter effective context, slower inference.</div>
"""))

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #a855f7; background:#faf5ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🧠 Step 9: From Tokens to Embeddings</h2>
</div>

After tokenization, each token ID maps to a **dense vector** via the embedding layer. These vectors live in a high-dimensional space where **meaning becomes geometry**:
- Similar concepts cluster together
- Relationships become directions (king - man + woman ≈ queen)
- Cosine similarity measures semantic relatedness

In [ ]:
# Ask models to explain embeddings
results = compare_models(
    "Explain word embeddings in 3-4 sentences. "
    "Include the famous king - man + woman = queen analogy "
    "and explain what cosine similarity measures.",
)
show_metrics_table(results)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📈 Step 10: Embedding Evolution</h2>
</div>

The evolution from static to contextual embeddings — like the move from ROM to RAM to CPU.

In [ ]:
# Embedding evolution timeline
from IPython.display import HTML, display

display(HTML("""
<table style="background:#f9fafb; border:1px solid #d1d5db; border-collapse:collapse; border-radius:8px; overflow:hidden; margin:10px 0; font-family:sans-serif; width:100%;">
  <thead><tr style="background:#1e3a5f;">
    <th style="padding:8px 12px; color:white; text-align:left;">Year</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Model</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Type</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Key Insight</th>
    <th style="padding:8px 12px; color:white; text-align:left;">Analogy</th>
  </tr></thead>
  <tbody>
    <tr><td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">2013</td>
        <td style="padding:6px 12px; font-weight:bold; border-bottom:1px solid #e5e7eb;">Word2Vec</td>
        <td style="padding:6px 12px; color:#f59e0b; border-bottom:1px solid #e5e7eb;">Static</td>
        <td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">Fixed lookup: \"bank\" = same vector always</td>
        <td style="padding:6px 12px; color:#6b7280; border-bottom:1px solid #e5e7eb;">ROM</td></tr>
    <tr><td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">2018</td>
        <td style="padding:6px 12px; font-weight:bold; border-bottom:1px solid #e5e7eb;">ELMo</td>
        <td style="padding:6px 12px; color:#2563eb; border-bottom:1px solid #e5e7eb;">Contextual</td>
        <td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">\"river bank\" \u2260 \"bank account\"</td>
        <td style="padding:6px 12px; color:#6b7280; border-bottom:1px solid #e5e7eb;">RAM</td></tr>
    <tr><td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">2018</td>
        <td style="padding:6px 12px; font-weight:bold; border-bottom:1px solid #e5e7eb;">BERT</td>
        <td style="padding:6px 12px; color:#2563eb; border-bottom:1px solid #e5e7eb;">Deep Contextual</td>
        <td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">Bidirectional attention, masked LM</td>
        <td style="padding:6px 12px; color:#6b7280; border-bottom:1px solid #e5e7eb;">CPU</td></tr>
    <tr><td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">2023+</td>
        <td style="padding:6px 12px; font-weight:bold; border-bottom:1px solid #e5e7eb;">BGE, E5, Nomic</td>
        <td style="padding:6px 12px; color:#16a34a; border-bottom:1px solid #e5e7eb;">Sentence-level</td>
        <td style="padding:6px 12px; border-bottom:1px solid #e5e7eb;">Contrastive training, optimized for retrieval</td>
        <td style="padding:6px 12px; color:#6b7280; border-bottom:1px solid #e5e7eb;">ASIC</td></tr>
  </tbody>
</table>
"""))
print("Modern embedding models (BGE, Nomic, GTE) are purpose-built for retrieval — they produce sentence-level vectors optimized for cosine similarity search.")

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🏆 Step 11: Knowledge Check</h2>
</div>

Let's test the models on tokenization pitfalls — a topic where reasoning quality really matters.

In [ ]:
# Pitfall question — models should catch the subtlety
results = compare_models(
    "I trained a model with one tokenizer, then at inference time I loaded a different tokenizer. "
    "The model runs without errors but produces garbage output. "
    "Explain why this happens in 2-3 sentences, referencing the embedding table.",
)
show_metrics_table(results)

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #22c55e; background:#f0fdf4; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">📝 Takeaways</h2>
</div>

- **Tokenization is the ADC of NLP** — it converts continuous text into discrete tokens. BPE builds vocabulary bottom-up by merging frequent pairs.
- **Vocabulary size is a tradeoff** — larger vocabularies produce shorter sequences (faster inference) but require more memory.
- **Fertility inequality** — English-centric tokenizers cost 3-5× more tokens for equivalent content in non-English languages.
- **Special tokens** (BOS, EOS, chat template markers) are first-class citizens — mishandling them causes hard-to-diagnose bugs.
- **Tokenizers are permanently bound to their model** — you cannot swap tokenizers after training.
- **Embeddings are learned coordinate systems** where meaning becomes geometry. Cosine similarity measures semantic relatedness.
- **Contextual embeddings** (BERT-era onward) produce different vectors for the same word in different contexts.
- Everything here ran locally — no data left your machine.

<hr style="border:none; height:3px; background:linear-gradient(90deg, #4f8ef7, #a855f7, #ec4899); border-radius:3px; margin:30px 0;">

<div style="border-left:6px solid #4f8ef7; background:#f0f4ff; padding:14px 20px; border-radius:0 8px 8px 0; margin:20px 0;">
  <h2 style="margin:0; color:#1a1a2e; font-size:1.4em;">🚀 What's Next</h2>
</div>

- **Section 03:** Prompting Techniques — zero-shot, few-shot, chain-of-thought, and more
- **Section 04:** Prompt Optimization — DSPy, automated optimization, prompt CI/CD
- **Section 05:** Structured Output — making LLMs return JSON you can actually parse